In [1]:
import os
from six import iteritems
from itertools import product
import pandas as pd

import logging

logger = logging.getLogger(__name__)

import pypsa

from add_electricity import load_costs, update_transmission_costs

idx = pd.IndexSlice

opt_name = {"Store": "e", "Line" : "s", "Transformer" : "s"}

In [2]:
display(pd.__version__)

'0.25.2'

In [3]:
def _add_indexed_rows(df, raw_index):
    new_index = df.index|pd.MultiIndex.from_product(raw_index)
    if isinstance(new_index, pd.Index):
        new_index = pd.MultiIndex.from_tuples(new_index)

    return df.reindex(new_index)

In [4]:
def assign_carriers(n):

    if "carrier" not in n.loads:
        n.loads["carrier"] = "electricity"
        for carrier in ["transport","heat","urban heat"]:
            n.loads.loc[n.loads.index.str.contains(carrier),"carrier"] = carrier

    n.storage_units['carrier'].replace({'hydro': 'hydro+PHS', 'PHS': 'hydro+PHS'}, inplace=True)

    if "carrier" not in n.lines:
        n.lines["carrier"] = "AC"

    n.lines["carrier"].replace({"AC": "lines"}, inplace=True)

    if n.links.empty: n.links["carrier"] = pd.Series(dtype=str)
    n.links["carrier"].replace({"DC": "lines"}, inplace=True)

    if "EU gas store" in n.stores.index and n.stores.loc["EU gas Store","carrier"] == "":
        n.stores.loc["EU gas Store","carrier"] = "gas Store"

In [5]:
def calculate_costs(n,label,costs):

    for c in n.iterate_components(n.branch_components|n.controllable_one_port_components^{"Load"}):
        capital_costs = c.df.capital_cost*c.df[opt_name.get(c.name,"p") + "_nom_opt"]
        capital_costs_grouped = capital_costs.groupby(c.df.carrier).sum()
        
        # Index tuple(s) indicating the newly to-be-added row(s)
        raw_index = tuple([[c.list_name],["capital"],list(capital_costs_grouped.index)])
        costs = _add_indexed_rows(costs, raw_index)

        costs.loc[idx[raw_index],label] = capital_costs_grouped.values

        if c.name == "Link":
            p = c.pnl.p0.multiply(n.snapshot_weightings,axis=0).sum()
        elif c.name == "Line":
            continue
        elif c.name == "StorageUnit":
            p_all = c.pnl.p.multiply(n.snapshot_weightings,axis=0)
            p_all[p_all < 0.] = 0.
            p = p_all.sum()
        else:
            p = c.pnl.p.multiply(n.snapshot_weightings,axis=0).sum()

        marginal_costs = p*c.df.marginal_cost

        marginal_costs_grouped = marginal_costs.groupby(c.df.carrier).sum()

        costs = costs.reindex(costs.index|pd.MultiIndex.from_product([[c.list_name],["marginal"],marginal_costs_grouped.index]))

        costs.loc[idx[c.list_name,"marginal",list(marginal_costs_grouped.index)],label] = marginal_costs_grouped.values

    return costs

In [6]:
def calculate_curtailment(n,label,curtailment):

    avail = n.generators_t.p_max_pu.multiply(n.generators.p_nom_opt).sum().groupby(n.generators.carrier).sum()
    used = n.generators_t.p.sum().groupby(n.generators.carrier).sum()

    curtailment[label] = (((avail - used)/avail)*100).round(3)

    return curtailment

In [7]:
def calculate_energy(n,label,energy):

    for c in n.iterate_components(n.one_port_components|n.branch_components):

        if c.name in n.one_port_components:
            c_energies = c.pnl.p.multiply(n.snapshot_weightings,axis=0).sum().multiply(c.df.sign).groupby(c.df.carrier).sum()
        else:
            c_energies = (-c.pnl.p1.multiply(n.snapshot_weightings,axis=0).sum() - c.pnl.p0.multiply(n.snapshot_weightings,axis=0).sum()).groupby(c.df.carrier).sum()

        energy = include_in_summary(energy, [c.list_name], label, c_energies)

    return energy

In [8]:
def include_in_summary(summary, multiindexprefix, label, item):

    # Index tuple(s) indicating the newly to-be-added row(s)
    raw_index = tuple([multiindexprefix,list(item.index)])
    summary = _add_indexed_rows(summary, raw_index)
    
    summary.loc[idx[raw_index], label] = item.values
    return summary

In [9]:
def calculate_capacity(n,label,capacity):

    for c in n.iterate_components(n.one_port_components):
        if 'p_nom_opt' in c.df.columns:
            c_capacities = abs(c.df.p_nom_opt.multiply(c.df.sign)).groupby(c.df.carrier).sum()
            capacity = include_in_summary(capacity, [c.list_name], label, c_capacities)

    for c in n.iterate_components(n.passive_branch_components):
        c_capacities = c.df['s_nom_opt'].groupby(c.df.carrier).sum()
        capacity = include_in_summary(capacity, [c.list_name], label, c_capacities)

    for c in n.iterate_components(n.controllable_branch_components):
        c_capacities = c.df.p_nom_opt.groupby(c.df.carrier).sum()
        capacity = include_in_summary(capacity, [c.list_name], label, c_capacities)

    return capacity

In [10]:
def calculate_supply(n,label,supply):
    """calculate the max dispatch of each component at the buses where the loads are attached"""

    load_types = n.loads.carrier.value_counts().index

    for i in load_types:

        buses = n.loads.bus[n.loads.carrier == i].values

        bus_map = pd.Series(False,index=n.buses.index)

        bus_map.loc[buses] = True

        for c in n.iterate_components(n.one_port_components):

            items = c.df.index[c.df.bus.map(bus_map)]

            if len(items) == 0:
                continue

            s = c.pnl.p[items].max().multiply(c.df.loc[items,'sign']).groupby(c.df.loc[items,'carrier']).sum()
            
            # Index tuple(s) indicating the newly to-be-added row(s)
            raw_index = tuple([[i],[c.list_name],list(s.index)])
            supply = _add_indexed_rows(supply, raw_index)
            
            supply.loc[idx[raw_index],label] = s.values


        for c in n.iterate_components(n.branch_components):

            for end in ["0","1"]:

                items = c.df.index[c.df["bus" + end].map(bus_map)]

                if len(items) == 0:
                    continue

                #lots of sign compensation for direction and to do maximums
                s = (-1)**(1-int(end))*((-1)**int(end)*c.pnl["p"+end][items]).max().groupby(c.df.loc[items,'carrier']).sum()

                supply = supply.reindex(supply.index|pd.MultiIndex.from_product([[i],[c.list_name],s.index]))
                supply.loc[idx[i,c.list_name,list(s.index)],label] = s.values

    return supply

In [11]:
def calculate_supply_energy(n,label,supply_energy):
    """calculate the total dispatch of each component at the buses where the loads are attached"""

    load_types = n.loads.carrier.value_counts().index

    for i in load_types:

        buses = n.loads.bus[n.loads.carrier == i].values

        bus_map = pd.Series(False,index=n.buses.index)

        bus_map.loc[buses] = True

        for c in n.iterate_components(n.one_port_components):

            items = c.df.index[c.df.bus.map(bus_map)]

            if len(items) == 0:
                continue

            s = c.pnl.p[items].sum().multiply(c.df.loc[items,'sign']).groupby(c.df.loc[items,'carrier']).sum()
          
            # Index tuple(s) indicating the newly to-be-added row(s)
            raw_index = tuple([[i],[c.list_name],list(s.index)])
            supply_energy = _add_indexed_rows(supply_energy, raw_index)
            
            supply_energy.loc[idx[raw_index],label] = s.values


        for c in n.iterate_components(n.branch_components):

            for end in ["0","1"]:

                items = c.df.index[c.df["bus" + end].map(bus_map)]

                if len(items) == 0:
                    continue

                s = (-1)*c.pnl["p"+end][items].sum().groupby(c.df.loc[items,'carrier']).sum()

                supply_energy = supply_energy.reindex(supply_energy.index|pd.MultiIndex.from_product([[i],[c.list_name],s.index]))
                supply_energy.loc[idx[i,c.list_name,list(s.index)],label] = s.values

    return supply_energy

In [12]:
def calculate_metrics(n,label,metrics):

    metrics = metrics.reindex(metrics.index|pd.Index(["line_volume","line_volume_limit","line_volume_AC","line_volume_DC","line_volume_shadow","co2_shadow"]))

    metrics.at["line_volume_DC",label] = (n.links.length*n.links.p_nom_opt)[n.links.carrier == "DC"].sum()
    metrics.at["line_volume_AC",label] = (n.lines.length*n.lines.s_nom_opt).sum()
    metrics.at["line_volume",label] = metrics.loc[["line_volume_AC","line_volume_DC"],label].sum()

    if hasattr(n,"line_volume_limit"):
        metrics.at["line_volume_limit",label] = n.line_volume_limit

    if hasattr(n,"line_volume_limit_dual"):
        metrics.at["line_volume_shadow",label] = n.line_volume_limit_dual

    if "CO2Limit" in n.global_constraints.index:
        metrics.at["co2_shadow",label] = n.global_constraints.at["CO2Limit","mu"]

    return metrics

In [13]:
def calculate_prices(n,label,prices):

    bus_type = pd.Series(n.buses.index.str[3:],n.buses.index).replace("","electricity")

    prices = prices.reindex(prices.index|bus_type.value_counts().index)

    #WARNING: this is time-averaged, should really be load-weighted average
    prices[label] = n.buses_t.marginal_price.mean().groupby(bus_type).mean()

    return prices



def calculate_weighted_prices(n,label,weighted_prices):
    # Warning: doesn't include storage units as loads


    weighted_prices = weighted_prices.reindex(pd.Index(["electricity","heat","space heat","urban heat","space urban heat","gas","H2"]))

    link_loads = {"electricity" :  ["heat pump", "resistive heater", "battery charger", "H2 Electrolysis"],
                  "heat" : ["water tanks charger"],
                  "urban heat" : ["water tanks charger"],
                  "space heat" : [],
                  "space urban heat" : [],
                  "gas" : ["OCGT","gas boiler","CHP electric","CHP heat"],
                  "H2" : ["Sabatier", "H2 Fuel Cell"]}

    for carrier in link_loads:

        if carrier == "electricity":
            suffix = ""
        elif carrier[:5] == "space":
            suffix = carrier[5:]
        else:
            suffix =  " " + carrier

        buses = n.buses.index[n.buses.index.str[2:] == suffix]

        if buses.empty:
            continue

        if carrier in ["H2","gas"]:
            load = pd.DataFrame(index=n.snapshots,columns=buses,data=0.)
        elif carrier[:5] == "space":
            load = heat_demand_df[buses.str[:2]].rename(columns=lambda i: str(i)+suffix)
        else:
            load = n.loads_t.p_set[buses]


        for tech in link_loads[carrier]:

            names = n.links.index[n.links.index.to_series().str[-len(tech):] == tech]

            if names.empty:
                continue

            load += n.links_t.p0[names].groupby(n.links.loc[names,"bus0"],axis=1).sum(axis=1)

        #Add H2 Store when charging
        if carrier == "H2":
            stores = n.stores_t.p[buses+ " Store"].groupby(n.stores.loc[buses+ " Store","bus"],axis=1).sum(axis=1)
            stores[stores > 0.] = 0.
            load += -stores

        weighted_prices.loc[carrier,label] = (load*n.buses_t.marginal_price[buses]).sum().sum()/load.sum().sum()

        if carrier[:5] == "space":
            print(load*n.buses_t.marginal_price[buses])

    return weighted_prices

In [14]:
# BROKEN don't use
#
# def calculate_market_values(n, label, market_values):
#     # Warning: doesn't include storage units

#     n.buses["suffix"] = n.buses.index.str[2:]
#     suffix = ""
#     buses = n.buses.index[n.buses.suffix == suffix]

#     ## First do market value of generators ##
#     generators = n.generators.index[n.buses.loc[n.generators.bus,"suffix"] == suffix]
#     techs = n.generators.loc[generators,"carrier"].value_counts().index
#     market_values = market_values.reindex(market_values.index | techs)

#     for tech in techs:
#         gens = generators[n.generators.loc[generators,"carrier"] == tech]
#         dispatch = n.generators_t.p[gens].groupby(n.generators.loc[gens,"bus"],axis=1).sum().reindex(columns=buses,fill_value=0.)
#         revenue = dispatch*n.buses_t.marginal_price[buses]
#         market_values.at[tech,label] = revenue.sum().sum()/dispatch.sum().sum()

#     ## Now do market value of links ##

#     for i in ["0","1"]:
#         all_links = n.links.index[n.buses.loc[n.links["bus"+i],"suffix"] == suffix]
#         techs = n.links.loc[all_links,"carrier"].value_counts().index
#         market_values = market_values.reindex(market_values.index | techs)

#         for tech in techs:
#             links = all_links[n.links.loc[all_links,"carrier"] == tech]
#             dispatch = n.links_t["p"+i][links].groupby(n.links.loc[links,"bus"+i],axis=1).sum().reindex(columns=buses,fill_value=0.)
#             revenue = dispatch*n.buses_t.marginal_price[buses]
#             market_values.at[tech,label] = revenue.sum().sum()/dispatch.sum().sum()

#     return market_values


# OLD CODE must be adapted

# def calculate_price_statistics(n, label, price_statistics):


#     price_statistics = price_statistics.reindex(price_statistics.index|pd.Index(["zero_hours","mean","standard_deviation"]))
#     n.buses["suffix"] = n.buses.index.str[2:]
#     suffix = ""
#     buses = n.buses.index[n.buses.suffix == suffix]

#     threshold = 0.1 #higher than phoney marginal_cost of wind/solar
#     df = pd.DataFrame(data=0.,columns=buses,index=n.snapshots)
#     df[n.buses_t.marginal_price[buses] < threshold] = 1.
#     price_statistics.at["zero_hours", label] = df.sum().sum()/(df.shape[0]*df.shape[1])
#     price_statistics.at["mean", label] = n.buses_t.marginal_price[buses].unstack().mean()
#     price_statistics.at["standard_deviation", label] = n.buses_t.marginal_price[buses].unstack().std()
#     return price_statistics

In [15]:
outputs = ["costs",
           "curtailment",
           "energy",
           "capacity",
           "supply",
           "supply_energy",
           "prices",
           "weighted_prices",
           # "price_statistics",
           # "market_values",
           "metrics",
           ]

def make_summaries(networks_dict, country='all'):

    
    # Empty dataframes for storing output
    columns = pd.MultiIndex.from_tuples(networks_dict.keys(),names=["simpl","clusters","ll","opts"])
    dfs = {}
    for output in outputs:
        dfs[output] = pd.DataFrame(columns=columns,dtype=float)

    for label, filename in iteritems(networks_dict):
        print(label, filename)
        
        # Try to load network from file
        if not os.path.exists(filename):
            print(f"Network {filename} does not exist on disk!!")
            continue
        try:
            n = pypsa.Network(filename)
            logger.info(f"Network {filename} loaded successfully.")
        except OSError:
            logger.warning(f"Unable to load network {filename} due to an error. Skipping.")
            continue
            
        # Restrict evaluation to some countries
        if country != 'all':
            n = n[n.buses.country == country]

        Nyears = n.snapshot_weightings.sum()/8760.
        costs = load_costs(Nyears, snakemake.input[0],
                           snakemake.config['costs'], snakemake.config['electricity'])
        
        # Update costs of transmission lines
        # modifies the network 'n'
        update_transmission_costs(n, costs, simple_hvdc_costs=False)
        
        # modifies the network 'n'
        assign_carriers(n)

        # Calculate and store the outputs in prepared dataframes
        for output in outputs:
            func = globals()[f"calculate_{output}"]
            dfs[output] = func(n, label, dfs[output])

    return dfs


def to_csv(dfs):
    dir = snakemake.output[0]
    os.makedirs(dir, exist_ok=True)
    for key, df in iteritems(dfs):
        df.to_csv(os.path.join(dir, f"{key}.csv"))

In [16]:
if __name__ == "__main__":
    # Detect running outside of snakemake and mock snakemake for testing
    
    rp = ''
    if "snakemake" not in globals():
        print("snakemake object not defined. Loading from pickle file.")
        import pickle

        with open("../snakemake.pickle", "rb") as pfile:
            snakemake = pickle.load(pfile)
        
        # Relative path to consider when not launching from pypsa-eur main folder
        rp = "../"
        snakemake.input  = [rp+p for p in snakemake.input]
        snakemake.output = [rp+p for p in snakemake.output]
    
    def expand_from_wildcard(key):
        w = getattr(snakemake.wildcards, key)
        return snakemake.config["scenario"][key] if w == "all" else [w]

    if snakemake.wildcards.ll.endswith("all"):
        ll = snakemake.config["scenario"]["ll"]
        if len(snakemake.wildcards.ll) == 4:
            ll = [l for l in ll if l[0] == snakemake.wildcards.ll[0]]
    else:
        ll = [snakemake.wildcards.ll]

    networks_dict = {(simpl,clusters,l,opts) : (rp+'results/networks/{network}_s{simpl}_{clusters}_l{ll}_{opts}.nc'
                                                 .format(network=snakemake.wildcards.network,
                                                         simpl=simpl,
                                                         clusters=clusters,
                                                         opts=opts,
                                                         ll=l))
                     for simpl in expand_from_wildcard("simpl")
                     for clusters in expand_from_wildcard("clusters")
                     for l in ll
                     for opts in expand_from_wildcard("opts")}

    print(networks_dict)
    

snakemake object not defined. Loading from pickle file.
{('', '64', 'vopt', 'Co2L-3H'): '../results/networks/elec_s_64_lvopt_Co2L-3H.nc'}


In [19]:
dfs = make_summaries(networks_dict, country=snakemake.wildcards.country)
 
to_csv(dfs)

('', '64', 'vopt', 'Co2L-3H') ../results/networks/elec_s_64_lvopt_Co2L-3H.nc


INFO:pypsa.io:Imported network elec_s_64_lvopt_Co2L-3H.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units
INFO:__main__:Network ../results/networks/elec_s_64_lvopt_Co2L-3H.nc loaded successfully.


In [23]:
dfs.keys()

dict_keys(['costs', 'curtailment', 'energy', 'capacity', 'supply', 'supply_energy', 'prices', 'weighted_prices', 'metrics'])

In [18]:
dfs['costs']

simpl                                          
clusters                                     64
ll                                         vopt
opts                                    Co2L-3H
generators    capital  OCGT        7.902287e+09
                       offwind-ac  8.709346e+10
                       offwind-dc  3.983500e+09
                       onwind      2.314283e+10
                       ror         8.908298e+09
                       solar       2.429852e+10
              marginal OCGT        9.438330e+09
                       offwind-ac  4.047496e+07
                       offwind-dc  1.706608e+06
                       onwind      1.218090e+07
                       ror         1.514675e+06
                       solar       1.026748e+07
lines         capital  lines       1.322225e+10
links         capital  lines       3.336426e+10
              marginal lines       0.000000e+00
storage_units capital  H2          8.447372e+09
                       battery     9.349806e+08
                       hydro+PHS   8.856679e+09
              marginal H2          6.556355e+05
                       battery     1.150300e+05
                       hydro+PHS   3.065869e+06

# --- End of original script ---

# +++ Testing range +++

In [15]:
# Load original snakemake config from pickled object
# This simulates snakemake running as for the real script
print("snakemake object not defined. Loading from pickle file.")
import pickle

with open("../snakemake.pickle", "rb") as pfile:
    snakemake = pickle.load(pfile)

# Relative path to consider when not launching from pypsa-eur main folder
rp = "../"
snakemake.input  = [rp+p for p in snakemake.input]
snakemake.output = [rp+p for p in snakemake.output]

def expand_from_wildcard(key):
    w = getattr(snakemake.wildcards, key)
    return snakemake.config["scenario"][key] if w == "all" else [w]

snakemake object not defined. Loading from pickle file.


In [16]:
display(snakemake.output)

['../results/summaries/elec_s_64_lvopt_Co2L-3H_all']

In [17]:
display(snakemake.input)

['../data/costs.csv', '../results/networks/elec_s_64_lvopt_Co2L-3H.nc']

In [18]:
if snakemake.wildcards.ll.endswith("all"):
    ll = snakemake.config["scenario"]["ll"]
    if len(snakemake.wildcards.ll) == 4:
        ll = [l for l in ll if l[0] == snakemake.wildcards.ll[0]]
else:
    ll = [snakemake.wildcards.ll]

In [19]:
display(ll)

['vopt']

In [20]:
display(snakemake.wildcards.network)

'elec'

In [21]:
networks_dict = {(simpl,clusters,l,opts) : (rp+'results/networks/{network}_s{simpl}_{clusters}_l{ll}_{opts}.nc'
                                             .format(network=snakemake.wildcards.network,
                                                     simpl=simpl,
                                                     clusters=clusters,
                                                     opts=opts,
                                                     ll=l))
                 for simpl in expand_from_wildcard("simpl")
                 for clusters in expand_from_wildcard("clusters")
                 for l in ll
                 for opts in expand_from_wildcard("opts")}

display(networks_dict)

{('',
  '64',
  'vopt',
  'Co2L-3H'): '../results/networks/elec_s_64_lvopt_Co2L-3H.nc'}

In [22]:
# Copy paste of the function make_summaries
# Reproduces this call:
# dfs = make_summaries(networks_dict, country=snakemake.wildcards.country)

# Function defaults
if "country" not in globals() or country is None:
    country = 'all'

# Inside the function
country = snakemake.wildcards.country

columns = pd.MultiIndex.from_tuples(networks_dict.keys(),names=["simpl","clusters","ll","opts"])

dfs = {}

# Prepare empty dataframes for storing the output
for output in outputs:
    dfs[output] = pd.DataFrame(columns=columns,dtype=float)

In [23]:
display(dfs)

{'costs': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: [], 'curtailment': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: [], 'energy': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: [], 'capacity': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: [], 'supply': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: [], 'supply_energy': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: [], 'prices': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: [], 'weighted_prices': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: [], 'metrics': Empty DataFrame
 Columns: [(, 64, vopt, Co2L-3H)]
 Index: []}

In [24]:
for label, filename in iteritems(networks_dict):
        print(label, filename)
        
        # Try to load network from file
        if not os.path.exists(filename):
            print(f"Network {filneame} does not exist on disk!!")
            continue
        try:
            n = pypsa.Network(filename)
            logger.info(f"Network {filename} loaded successfully.")
        except OSError:
            logger.warning(f"Unable to load network {filename} due to an error. Skipping.")
            continue
            
        # Restrict evaluation to some countries
        if country != 'all':
            n = n[n.buses.country == country]

        Nyears = n.snapshot_weightings.sum()/8760.
        costs = load_costs(Nyears, snakemake.input[0],
                           snakemake.config['costs'], snakemake.config['electricity'])
        
        # Update costs of transmission lines
        # modifies the network 'n'
        update_transmission_costs(n, costs, simple_hvdc_costs=False)
        
        # modifies the network 'n'
        assign_carriers(n)
        break

        # Calculate and store the outputs in prepared dataframes
        for output in outputs:
            func = locals()[f"calculate_{output}"]
            dfs[output] = func(n, label, dfs[output])

('', '64', 'vopt', 'Co2L-3H') ../results/networks/elec_s_64_lvopt_Co2L-3H.nc


INFO:pypsa.io:Imported network elec_s_64_lvopt_Co2L-3H.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units
INFO:__main__:Network ../results/networks/elec_s_64_lvopt_Co2L-3H.nc loaded successfully.


In [25]:
costs

parameter,co2_emissions,FOM,VOM,discount rate,efficiency,fuel,investment,lifetime,capital_cost,marginal_cost
technology,,,,,,,,,,
CCGT,0.187,2.500000,4.00,0.07,0.500000,21.6,800000.00,30.0,84469.122809,47.200000
DAC,0.000,4.000000,0.00,0.07,1.000000,0.0,250.00,30.0,30.146601,0.000000
HVAC overhead,0.000,2.000000,0.00,0.07,1.000000,0.0,400.00,40.0,38.003656,0.000000
HVDC inverter pair,0.000,2.000000,0.00,0.07,1.000000,0.0,150000.00,40.0,14251.370831,0.000000
HVDC overhead,0.000,2.000000,0.00,0.07,1.000000,0.0,400.00,40.0,38.003656,0.000000
HVDC submarine,0.000,2.000000,0.00,0.07,1.000000,0.0,2000.00,40.0,190.018278,0.000000
OCGT,0.187,3.750000,3.00,0.07,0.390000,21.6,400000.00,30.0,47234.561404,58.384615
PHS,0.000,1.000000,0.00,0.07,0.750000,0.0,2000000.00,80.0,160627.143522,0.000000
battery inverter,0.000,3.000000,0.00,0.07,0.810000,0.0,309565.20,20.0,38507.720936,0.000000


In [26]:
costs = dfs['costs']
costs.index

Index([], dtype='object')

In [27]:
def _add_indexed_rows(df, raw_index):
    new_index = df.index|pd.MultiIndex.from_product(raw_index)
    if isinstance(new_index, pd.Index):
        new_index = pd.MultiIndex.from_tuples(new_index)

    return df.reindex(new_index)

In [28]:
for c in n.iterate_components(n.branch_components|n.controllable_one_port_components^{"Load"}):
        capital_costs = c.df.capital_cost*c.df[opt_name.get(c.name,"p") + "_nom_opt"]
        capital_costs_grouped = capital_costs.groupby(c.df.carrier).sum()

        display([costs.index,[c.list_name],["capital"],capital_costs_grouped.index])
        display(costs.index|pd.MultiIndex.from_product([[c.list_name],["capital"],capital_costs_grouped.index]))
        
        # Index tuple(s) indicating the newly to-be-added row(s)
        raw_index = ([c.list_name],["capital"],list(capital_costs_grouped.index))
        costs = _add_indexed_rows(costs, raw_index)

        display(costs)
        costs.loc[idx[raw_index],label] = capital_costs_grouped.values

        if c.name == "Link":
            p = c.pnl.p0.multiply(n.snapshot_weightings,axis=0).sum()
        elif c.name == "Line":
            continue
        elif c.name == "StorageUnit":
            p_all = c.pnl.p.multiply(n.snapshot_weightings,axis=0)
            p_all[p_all < 0.] = 0.
            p = p_all.sum()
        else:
            p = c.pnl.p.multiply(n.snapshot_weightings,axis=0).sum()

        marginal_costs = p*c.df.marginal_cost

        marginal_costs_grouped = marginal_costs.groupby(c.df.carrier).sum()

        costs = costs.reindex(costs.index|pd.MultiIndex.from_product([[c.list_name],["marginal"],marginal_costs_grouped.index]))

        costs.loc[idx[c.list_name,"marginal",list(marginal_costs_grouped.index)],label] = marginal_costs_grouped.values


[Index([], dtype='object'),
 ['links'],
 ['capital'],
 Index(['lines'], dtype='object', name='carrier')]

Index([('links', 'capital', 'lines')], dtype='object')

,,simpl,
,,clusters,64
,,ll,vopt
,,opts,Co2L-3H
links,capital,lines,NaN


[MultiIndex([('links',  'capital', 'lines'),
             ('links', 'marginal', 'lines')],
            ),
 ['generators'],
 ['capital'],
 Index(['OCGT', 'offwind-ac', 'offwind-dc', 'onwind', 'ror', 'solar'], dtype='object', name='carrier')]

MultiIndex([('generators',  'capital',       'OCGT'),
            ('generators',  'capital', 'offwind-ac'),
            ('generators',  'capital', 'offwind-dc'),
            ('generators',  'capital',     'onwind'),
            ('generators',  'capital',        'ror'),
            ('generators',  'capital',      'solar'),
            (     'links',  'capital',      'lines'),
            (     'links', 'marginal',      'lines')],
           )

...

MultiIndex([(   'generators',  'capital',       'OCGT'),
            (   'generators',  'capital', 'offwind-ac'),
            (   'generators',  'capital', 'offwind-dc'),
            (   'generators',  'capital',     'onwind'),
            (   'generators',  'capital',        'ror'),
            (   'generators',  'capital',      'solar'),
            (   'generators', 'marginal',       'OCGT'),
            (   'generators', 'marginal', 'offwind-ac'),
            (   'generators', 'marginal', 'offwind-dc'),
            (   'generators', 'marginal',     'onwind'),
            (   'generators', 'marginal',        'ror'),
            (   'generators', 'marginal',      'solar'),
            (        'links',  'capital',      'lines'),
            (        'links', 'marginal',      'lines'),
            ('storage_units',  'capital',         'H2'),
            ('storage_units',  'capital',    'battery'),
            ('storage_units',  'capital',  'hydro+PHS')],
           )

simpl                                          
clusters                                     64
ll                                         vopt
opts                                    Co2L-3H
generators    capital  OCGT        7.902287e+09
                       offwind-ac  8.709346e+10
                       offwind-dc  3.983500e+09
                       onwind      2.314283e+10
                       ror         8.908298e+09
                       solar       2.429852e+10
              marginal OCGT        9.438330e+09
                       offwind-ac  4.047496e+07
                       offwind-dc  1.706608e+06
                       onwind      1.218090e+07
                       ror         1.514675e+06
                       solar       1.026748e+07
links         capital  lines       3.336426e+10
              marginal lines       0.000000e+00
storage_units capital  H2                   NaN
                       battery              NaN
                       hydro+PHS            NaN

[MultiIndex([(   'generators',  'capital',       'OCGT'),
             (   'generators',  'capital', 'offwind-ac'),
             (   'generators',  'capital', 'offwind-dc'),
             (   'generators',  'capital',     'onwind'),
             (   'generators',  'capital',        'ror'),
             (   'generators',  'capital',      'solar'),
             (   'generators', 'marginal',       'OCGT'),
             (   'generators', 'marginal', 'offwind-ac'),
             (   'generators', 'marginal', 'offwind-dc'),
             (   'generators', 'marginal',     'onwind'),
             (   'generators', 'marginal',        'ror'),
             (   'generators', 'marginal',      'solar'),
             (        'links',  'capital',      'lines'),
             (        'links', 'marginal',      'lines'),
             ('storage_units',  'capital',         'H2'),
             ('storage_units',  'capital',    'battery'),
             ('storage_units',  'capital',  'hydro+PHS'),
             (

MultiIndex([(   'generators',  'capital',       'OCGT'),
            (   'generators',  'capital', 'offwind-ac'),
            (   'generators',  'capital', 'offwind-dc'),
            (   'generators',  'capital',     'onwind'),
            (   'generators',  'capital',        'ror'),
            (   'generators',  'capital',      'solar'),
            (   'generators', 'marginal',       'OCGT'),
            (   'generators', 'marginal', 'offwind-ac'),
            (   'generators', 'marginal', 'offwind-dc'),
            (   'generators', 'marginal',     'onwind'),
            (   'generators', 'marginal',        'ror'),
            (   'generators', 'marginal',      'solar'),
            (        'lines',  'capital',      'lines'),
            (        'links',  'capital',      'lines'),
            (        'links', 'marginal',      'lines'),
            ('storage_units',  'capital',         'H2'),
            ('storage_units',  'capital',    'battery'),
            ('storage_units',  

simpl                                          
clusters                                     64
ll                                         vopt
opts                                    Co2L-3H
generators    capital  OCGT        7.902287e+09
                       offwind-ac  8.709346e+10
                       offwind-dc  3.983500e+09
                       onwind      2.314283e+10
                       ror         8.908298e+09
                       solar       2.429852e+10
              marginal OCGT        9.438330e+09
                       offwind-ac  4.047496e+07
                       offwind-dc  1.706608e+06
                       onwind      1.218090e+07
                       ror         1.514675e+06
                       solar       1.026748e+07
lines         capital  lines                NaN
links         capital  lines       3.336426e+10
              marginal lines       0.000000e+00
storage_units capital  H2          8.447372e+09
                       battery     9.349806e+08
                       hydro+PHS   8.856679e+09
              marginal H2          6.556355e+05
                       battery     1.150300e+05
                       hydro+PHS   3.065869e+06

In [29]:
costs

simpl                                          
clusters                                     64
ll                                         vopt
opts                                    Co2L-3H
generators    capital  OCGT        7.902287e+09
                       offwind-ac  8.709346e+10
                       offwind-dc  3.983500e+09
                       onwind      2.314283e+10
                       ror         8.908298e+09
                       solar       2.429852e+10
              marginal OCGT        9.438330e+09
                       offwind-ac  4.047496e+07
                       offwind-dc  1.706608e+06
                       onwind      1.218090e+07
                       ror         1.514675e+06
                       solar       1.026748e+07
lines         capital  lines       1.322225e+10
links         capital  lines       3.336426e+10
              marginal lines       0.000000e+00
storage_units capital  H2          8.447372e+09
                       battery     9.349806e+08
                       hydro+PHS   8.856679e+09
              marginal H2          6.556355e+05
                       battery     1.150300e+05
                       hydro+PHS   3.065869e+06

In [67]:
tuple(raw_index)

(['storage_units'], ['capital'], ['H2', 'battery', 'hydro+PHS'])

In [69]:
pd.MultiIndex.from_product(tuple(raw_index))

MultiIndex([('storage_units', 'capital',        'H2'),
            ('storage_units', 'capital',   'battery'),
            ('storage_units', 'capital', 'hydro+PHS')],
           )

In [70]:
pd.MultiIndex.from_product(raw_index)

MultiIndex([('storage_units', 'capital',        'H2'),
            ('storage_units', 'capital',   'battery'),
            ('storage_units', 'capital', 'hydro+PHS')],
           )

In [54]:
idx[[c.list_name],["capital"],list(capital_costs_grouped.index)]

(['storage_units'], ['capital'], ['H2', 'battery', 'hydro+PHS'])

In [51]:
idx[raw_index]

[['storage_units'], ['capital'], ['H2', 'battery', 'hydro+PHS']]

In [65]:
costs.loc[idx[tuple(raw_index)],label] = capital_costs_grouped.values

In [66]:
costs

simpl                                        
clusters                                   64
ll                                       vopt
opts                                  Co2L-3H
storage_units capital H2         8.447372e+09
                      battery    9.349806e+08
                      hydro+PHS  8.856679e+09

In [49]:
costs.loc['storage_units', "capital"]

simpl,
clusters,64
ll,vopt
opts,Co2L-3H
H2,NaN
battery,NaN
hydro+PHS,NaN


In [50]:
costs.loc[idx[[c.list_name],["capital"],list(capital_costs_grouped.index)],label]

storage_units  capital  H2          NaN
                        battery     NaN
                        hydro+PHS   NaN
Name: (, 64, vopt, Co2L-3H), dtype: float64

In [72]:
costs.loc[idx[c.list_name,"capital",list(capital_costs_grouped.index)],label]

lines  capital  lines    1.322225e+10
Name: (, 64, vopt, Co2L-3H), dtype: float64

In [75]:
pd.MultiIndex.from_product([c.list_name,["capital"],list(capital_costs_grouped.index)])

TypeError: Input must be list-like